In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import col, sum, avg, rank, dense_rank, row_number, lit
from pyspark.sql.window import Window

In [0]:
movies_df = (
    spark.read
        .table("movie_silver.movies")
        .filter(f"file_date = '{v_file_date}'")
).select("movie_id", "budget", "revenue", "year_release_date")

In [0]:
countries_df = (
    spark.read
        .table("movie_silver.countries")
).select("country_id", "country_name")

In [0]:
production_countries_df = (
    spark.read
        .table("movie_silver.production_countries")
        .filter(col("file_date") == v_file_date)
)


In [0]:
rank_budget_revenue = (
    Window.partitionBy("year_release_date").
        orderBy(
            col("total_budget").desc(), col("total_revenue").desc()
            )
)

results_group_movie_country = (
    movies_df.filter(movies_df.year_release_date >= 2015)
        .join(production_countries_df,
              movies_df["movie_id"] == production_countries_df["movie_id"], "inner")
        .join(countries_df,
              production_countries_df["country_id"] == countries_df["country_id"], "inner")
        .select(
            "year_release_date",
            "country_name",
            "budget",
            "revenue"
        ).groupBy("year_release_date", "country_name")
            .agg(
                sum("budget").alias("total_budget"),
                sum("revenue").alias("total_revenue")
        )
        .orderBy(col("total_budget").desc())
        .withColumn("budget_rank",
                    row_number()
                        .over(Window.partitionBy("year_release_date")
                              .orderBy(col("total_budget").desc())))
        .withColumn("rank_budget_revenue",
                    dense_rank().over(rank_budget_revenue))
        .withColumn("created_date", lit(v_file_date))
)
display(results_group_movie_country)

In [0]:
overwrite_partitioned_table("movie_silver", "results_group_movie_country","created_date", v_file_date)

In [0]:
(
    results_group_movie_country
        .write
            .mode("append")
            .saveAsTable(f"movie_gold.results_group_movie_country")
)